In [2]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import skew, kurtosis

In [3]:
label_df = pd.read_csv(r"PHQ8_labels.csv")
display(label_df)

,Participant_ID,PHQ8_Binary,PHQ8_Score
0,300,0,2
1,301,0,3
2,302,0,4
3,303,0,0
4,304,0,6
...,...,...,...
184,488,0,0
185,489,0,3
186,490,0,2
187,491,0,8


In [4]:
# total distribution of labels: 1 - depressed | 0 - not depressed
print(label_df["PHQ8_Binary"].value_counts())

PHQ8_Binary
0    133
1     56
Name: count, dtype: int64


In [5]:
participant_ids = list(label_df["Participant_ID"])

In [6]:
def compute_statistics(window):
    stats = {}
    for col in window.columns:
        if window[col].dtype in ['int64', 'float64']:  # Numeric columns
            stats[f'{col}_mean'] = window[col].mean()
            stats[f'{col}_std'] = window[col].std()
            stats[f'{col}_min'] = window[col].min()
            stats[f'{col}_max'] = window[col].max()
            stats[f'{col}_sum'] = window[col].sum()
            stats[f'{col}_median'] = window[col].median()
            stats[f'{col}_var'] = window[col].var()
            #stats[f'{col}_skew'] = skew(window[col].dropna())
            #stats[f'{col}_kurt'] = kurtosis(window[col].dropna())
    return pd.Series(stats)


def sliding_window(df, window_size, step_size):
    aggregated_windows = []
    for start in range(0, len(df) - window_size + 1, step_size):
        end = start + window_size
        window = df.iloc[start:end]
        aggregated_window = compute_statistics(window)
        aggregated_windows.append(aggregated_window)
    return pd.DataFrame(aggregated_windows)

In [7]:
window_size = 30  # For 1 second of data, assuming 30 rows per second
step_size = 30    # Adjust as needed


In [8]:
data_path = r"datasets\DAIC-Woz"
#participant_id = 303
AUs_df = pd.DataFrame()


for participant_id in participant_ids:
    print(participant_id)
    temp_df = pd.DataFrame()
    file = os.path.join(data_path, f"{participant_id}_P", f"{participant_id}_CLNF_AUs.txt")
    if os.path.exists(file):
        # Retrieve the PHQ8_Binary value for this participant from train_df
        phq8_binary_value = label_df.loc[label_df['Participant_ID'] == participant_id, 'PHQ8_Binary'].values[0]

    #print(len(phq8_binary_value))

        temp = pd.read_csv(file)
        temp.drop(columns=["frame", " timestamp", " confidence", " success"], inplace = True)
        temp.columns = ["AU01_r", "AU02_r", "AU04_r", "AU05_r", "AU06_r", "AU09_r", "AU10_r", "AU12_r", "AU14_r", "AU15_r", "AU17_r", "AU20_r", 
                        "AU25_r", "AU26_r", "AU04_c", "AU12_c",	"AU15_c", "AU23_c", "AU28_c", "AU45_c"]
        
        temp_df = sliding_window(temp, window_size, step_size)

        temp_df["Target"] = phq8_binary_value

        AUs_df = pd.concat([AUs_df, temp_df], ignore_index = True, axis=0)

300
301
302
303
304
305
306
307
308
309
310
311
312
313
314
315
316
317
318
319
320
321
322
323
324
325
326
327
328
329
330
331
332
333
334
335
336
337
338
339
340
341
343
344
345
346
347
348
349
350
351
352
353
354
355
356
357
358
359
360
361
362
363
364
365
366
367
368
369
370
371
372
373
374
375
376
377
378
379
380
381
382
383
384
385
386
387
388
389
390
391
392
393
395
396
397
399
400
401
402
403
404
405
406
407
408
409
410
411
412
413
414
415
416
417
418
419
420
421
422
423
424
425
426
427
428
429
430
431
432
433
434
435
436
437
438
439
440
441
442
443
444
445
446
447
448
449
450
451
452
453
454
455
456
457
458
459
461
462
463
464
465
466
467
468
469
470
471
472
473
474
475
476
477
478
479
480
481
482
483
484
485
486
487
488
489
490
491
492


In [10]:
display(AUs_df)

,AU01_r_mean,AU01_r_std,AU01_r_min,AU01_r_max,AU01_r_sum,AU01_r_median,AU01_r_var,AU02_r_mean,AU02_r_std,AU02_r_min,...,AU28_c_median,AU28_c_var,AU45_c_mean,AU45_c_std,AU45_c_min,AU45_c_max,AU45_c_sum,AU45_c_median,AU45_c_var,Target
0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.318062,0.394306,0.0,...,0.0,0.000000,0.466667,0.507416,0.0,1.0,14.0,0.0,0.257471,0
1,0.856076,0.506571,0.000000,2.01038,25.682282,0.855557,0.256614,0.229474,0.340772,0.0,...,0.0,0.000000,0.733333,0.449776,0.0,1.0,22.0,1.0,0.202299,0
2,1.443779,0.374480,0.695651,2.08513,43.313363,1.574520,0.140235,0.105713,0.282239,0.0,...,0.0,0.000000,1.000000,0.000000,1.0,1.0,30.0,1.0,0.000000,0
3,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-100.0,0.000000,-100.000000,0.000000,-100.0,-100.0,-3000.0,-100.0,0.000000,0
4,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,-100.0,0.000000,-100.000000,0.000000,-100.0,-100.0,-3000.0,-100.0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180576,1.256444,0.367732,0.666390,1.89970,37.693324,1.223640,0.135227,0.297032,0.355202,0.0,...,0.0,0.240230,0.700000,0.466092,0.0,1.0,21.0,1.0,0.217241,0
180577,0.936931,0.144699,0.681188,1.32349,28.107944,0.922400,0.020938,0.182649,0.361233,0.0,...,0.0,0.165517,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0
180578,1.007713,0.173860,0.696032,1.36007,30.231381,1.005124,0.030227,0.596033,0.359511,0.0,...,0.0,0.000000,0.066667,0.253708,0.0,1.0,2.0,0.0,0.064368,0
180579,0.671213,0.413603,0.000000,1.49013,20.136401,0.707252,0.171067,0.498766,0.378688,0.0,...,0.0,0.000000,0.133333,0.345746,0.0,1.0,4.0,0.0,0.119540,0


In [11]:
AUs_df.isna().sum().sum()

0

In [12]:
print(AUs_df["Target"].value_counts()/len(AUs_df))
print(label_df["PHQ8_Binary"].value_counts()/len(label_df))

Target
0    0.689486
1    0.310514
Name: count, dtype: float64
PHQ8_Binary
0    0.703704
1    0.296296
Name: count, dtype: float64


In [13]:
AUs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180581 entries, 0 to 180580
Columns: 141 entries, AU01_r_mean to Target
dtypes: float64(140), int64(1)
memory usage: 194.3 MB


In [ ]:
# Get a list of all columns except 'Target'
columns = [col for col in AUs_df.columns if col != 'Target']

# Set up the matplotlib figure
plt.figure(figsize=(15, 10))

# Plot each column
for i, column in enumerate(columns, 1):
    plt.subplot(5, 4, i)  # Adjust the number of rows and columns to fit all plots
    sns.boxplot(y=AUs_df[column])
    plt.title(column)

# Adjust layout to prevent overlap
plt.tight_layout()

# Display the plots
plt.show()

In [14]:
# Exclude the 'Target' column for normalization
columns = [col for col in AUs_df.columns if col != 'Target']

# Normalize the data
scaler = MinMaxScaler()
AUs_df_normalized = AUs_df.copy()
AUs_df_normalized[columns] = scaler.fit_transform(AUs_df[columns])

# Set up the matplotlib figure
#plt.figure(figsize=(15, 10))

# Plot each column
#for i, column in enumerate(columns, 1):
#    plt.subplot(5, 4, i)  # Adjust the number of rows and columns to fit all plots
#    sns.boxplot(y=AUs_df_normalized[column])
#    plt.title(column)

# Adjust layout to prevent overlap
#plt.tight_layout()

# Display the plots
#plt.show()

In [15]:
def remove_outliers_iqr(df, columns):
    df_cleaned = df.copy()
    
    for column in columns:
        # Calculate Q1 (25th percentile) and Q3 (75th percentile)
        Q1 = df_cleaned[column].quantile(0.25)
        Q3 = df_cleaned[column].quantile(0.75)
        IQR = Q3 - Q1
        
        # Define bounds
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Filter out the outliers
        df_cleaned = df_cleaned[(df_cleaned[column] >= lower_bound) & (df_cleaned[column] <= upper_bound)]
    
    return df_cleaned


In [16]:
# Remove outliers
AUs_df_cleaned = remove_outliers_iqr(AUs_df, columns)

In [17]:
AUs_df_cleaned.shape

(4166, 141)

In [ ]:
# Set up the matplotlib figure
plt.figure(figsize=(15, 10))

# Plot each column
for i, column in enumerate(columns, 1):
    plt.subplot(5, 4, i)  # Adjust the number of rows and columns to fit all plots
    sns.boxplot(y=AUs_df_cleaned[column])
    plt.title(column)

# Adjust layout to prevent overlap
plt.tight_layout()

# Display the plots
plt.show()

In [18]:
print("original:", AUs_df.shape)
print("after normalize:", AUs_df_normalized.shape)
print("after outlier removal:", AUs_df_cleaned.shape)

original: (180581, 141)
after normalize: (180581, 141)
after outlier removal: (4166, 141)


In [19]:
AUs_df.to_csv("AUs_df_combined_original.csv", index = False)


In [20]:
AUs_df_normalized.to_csv("AUs_df_normal.csv", index = False)


In [21]:
AUs_df_cleaned.to_csv("AUs_df_cleaned.csv", index = False)